## Ingesting the documents

In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader

vrmuseum_docs = CSVLoader(
    file_path='../../data/processed/vrmuseum.csv',
    content_columns=['content'],
    metadata_columns=['title'],
    encoding='utf-8'
).load()

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 256
chunk_overlap = 32

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
doc_splits = text_splitter.split_documents(vrmuseum_docs)

In [9]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

model_name = 'Qwen/Qwen3-Embedding-0.6B'
embeddings = HuggingFaceEmbeddings(model_name=model_name)

vector_store = Chroma(
    collection_name='main', 
    embedding_function=embeddings,
    persist_directory='../../embeddings/vrmuseum'
)

In [11]:
from uuid import uuid4

def split_list(input_list, chunk_size=5461):
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

for batch_docs in split_list(doc_splits):
    batch_ids = [str(uuid4()) for _ in range(len(doc_splits))]
    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

In [15]:
dense_retriever = vector_store.as_retriever(search_kwargs={'k': 5})

In [ ]:
# Test retrieval of five results
results = dense_retriever.invoke('preservation')

for i, doc in enumerate(results):
    name = doc.page_content.split('\n')[0].split('title: ')[-1]
    print(f'===== RESULT {i  + 1}: {name} =====')
    print(doc.page_content)

===== RESULT 1: Ibis mummy? =====
content: title: Ibis mummy?
scientific name: nan
located: Redpath Museum
RM 2727.2
information: 
What is inside this bundle? A CT scan revealed that this bundle was 
only filled with rags and that nothing was actually embalmed inside! This is an example of a ‘pseudo-mummy’. Considering that it looks so similar to a real ibis mummy, it may well have been a key component of one of the oldest documented scams in human history!  
Real or fake? In a story published in the National Geographic magazine in 2008, zooarcheologist Salima Ikram notes a trend that might not be so surprising to a 21st-century human: the more attractive the bundle, the more likely it is to contain a fake mummy!  
The real ibis mummy and the fake mummy together present an exception to this rule: the wrapping of the real mummy features a beautiful artistic depiction of an ibis, while that of the fake mummy has no decoration at all. 
Do we know exactly where this mummy was made? We are 